In [4]:
import pandas as pd 
import numpy as np 

In [5]:
# 1. Customers DataFrame
customers_data = {
    'customer_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 'Heidi'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', 'diana@email.com', 
              'eve@email.com', 'frank@email.com', 'grace@email.com', 'heidi@email.com'],
    'city': ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'New York', 'Los Angeles', 'Chicago'],
    'signup_date': ['2023-01-15', '2023-02-20', '2023-03-10', '2023-04-05', '2023-05-12', '2023-06-18', '2023-07-22', '2023-08-30']
}
customers = pd.DataFrame(customers_data)
customers['signup_date'] = pd.to_datetime(customers['signup_date'])

# 2. Movies DataFrame
movies_data = {
    'movie_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'title': ['The Matrix', 'Inception', 'Toy Story', 'The Godfather', 'Pulp Fiction', 
              'The Dark Knight', 'Spirited Away', 'Parasite', 'Interstellar', 'The Room'],
    'genre': ['Sci-Fi', 'Sci-Fi', 'Animation', 'Crime', 'Crime', 'Action', 'Animation', 'Thriller', 'Sci-Fi', 'Drama'],
    'release_year': [1999, 2010, 1995, 1972, 1994, 2008, 2001, 2019, 2014, 2003],
    'rating': [5, 4, 5, 5, 4, 5, 5, 5, 4, 2] # 'The Room' is rated below 3 for our queries
}
movies = pd.DataFrame(movies_data)

# 3. Rentals DataFrame
rentals_data = {
    'rental_id': range(1, 26),
    'customer_id': [1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 5, 5, 5, 6, 6, 7, 7, 8, 8, 1, 2, 3, 4, 5, 8],
    'movie_id': [101, 102, 103, 101, 110, 104, 105, 106, 107, 108, 109, 110, 101, 102, 103, 104, 105, 106, 107, 109, 104, 108, 110, 102, 101],
    'rental_date': ['2023-09-01'] * 25,
    'return_date': ['2023-09-05'] * 25,
    'price': [2.99, 3.99, 1.99, 2.99, 1.99, 3.99, 2.99, 3.99, 2.99, 3.99, 3.99, 1.99, 2.99, 3.99, 1.99, 3.99, 2.99, 3.99, 2.99, 3.99, 3.99, 3.99, 1.99, 3.99, 2.99]
}
rentals = pd.DataFrame(rentals_data)
rentals['rental_date'] = pd.to_datetime(rentals['rental_date'])
rentals['return_date'] = pd.to_datetime(rentals['return_date'])

In [6]:
# 1. Find all movies rented by a specific customer (e.g., Alice, customer_id = 1)
alice_rentals = rentals[rentals['customer_id'] == 1].merge(movies, on='movie_id')
print("Movies rented by Alice:\n", alice_rentals[['title', 'genre', 'rental_date']])

# 2. Find the top 5 most-rented movies
top_5_movies = rentals.groupby('movie_id').size().reset_index(name='rental_count')
top_5_movies = top_5_movies.merge(movies, on='movie_id').sort_values(by='rental_count', ascending=False).head(5)
print("\nTop 5 Most Rented Movies:\n", top_5_movies[['title', 'rental_count']])

# 3. Compute the total revenue per genre
revenue_per_genre = rentals.merge(movies, on='movie_id').groupby('genre')['price'].sum().reset_index()
revenue_per_genre = revenue_per_genre.sort_values(by='price', ascending=False)
print("\nTotal Revenue per Genre:\n", revenue_per_genre)

# 4. Find customers who have never rented a movie rated below 3
# First, identify customers who HAVE rented a badly-rated movie
rentals_with_movies = rentals.merge(movies, on='movie_id')
customers_with_bad_rentals = rentals_with_movies[rentals_with_movies['rating'] < 3]['customer_id'].unique()

# Then, filter the customers dataframe to exclude those IDs
elite_customers = customers[~customers['customer_id'].isin(customers_with_bad_rentals)]
print("\nCustomers who never rented a movie rated below 3:\n", elite_customers[['name', 'email']])

Movies rented by Alice:
           title      genre rental_date
0    The Matrix     Sci-Fi  2023-09-01
1     Inception     Sci-Fi  2023-09-01
2     Toy Story  Animation  2023-09-01
3  Interstellar     Sci-Fi  2023-09-01

Top 5 Most Rented Movies:
            title  rental_count
0     The Matrix             4
1      Inception             3
3  The Godfather             3
9       The Room             3
2      Toy Story             2

Total Revenue per Genre:
        genre  price
4     Sci-Fi  31.91
2      Crime  17.95
1  Animation   9.96
0     Action   7.98
5   Thriller   7.98
3      Drama   5.97

Customers who never rented a movie rated below 3:
       name              email
0    Alice    alice@email.com
2  Charlie  charlie@email.com
5    Frank    frank@email.com
6    Grace    grace@email.com
7    Heidi    heidi@email.com


In [7]:
# Create a single denormalized table
denormalized_df = rentals.merge(customers, on='customer_id').merge(movies, on='movie_id')

# Displaying a subset of columns to highlight the redundancy
print(denormalized_df[['rental_id', 'name', 'email', 'title', 'genre', 'price']])

    rental_id     name              email            title      genre  price
0           1    Alice    alice@email.com       The Matrix     Sci-Fi   2.99
1           2    Alice    alice@email.com        Inception     Sci-Fi   3.99
2           3    Alice    alice@email.com        Toy Story  Animation   1.99
3           4      Bob      bob@email.com       The Matrix     Sci-Fi   2.99
4           5      Bob      bob@email.com         The Room      Drama   1.99
5           6  Charlie  charlie@email.com    The Godfather      Crime   3.99
6           7  Charlie  charlie@email.com     Pulp Fiction      Crime   2.99
7           8  Charlie  charlie@email.com  The Dark Knight     Action   3.99
8           9    Diana    diana@email.com    Spirited Away  Animation   2.99
9          10    Diana    diana@email.com         Parasite   Thriller   3.99
10         11      Eve      eve@email.com     Interstellar     Sci-Fi   3.99
11         12      Eve      eve@email.com         The Room      Drama   1.99

when creating denormalized, we encounter may duplicates for users, and if we want to change something about any user, we have to change multiple rows. And that wastes time and storage

In [8]:
import json

# Define the data using a document model approach
# Each document is a dictionary representing a customer and their rental history
customer_documents = [
    {
        "customer_id": 1, "name": "Alice", "email": "alice@email.com", "city": "New York", "signup_date": "2023-01-15",
        "rentals": [
            {"rental_id": 1, "movie_id": 101, "title": "The Matrix", "genre": "Sci-Fi", "rating": 5, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 2, "movie_id": 102, "title": "Inception", "genre": "Sci-Fi", "rating": 4, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 3, "movie_id": 103, "title": "Toy Story", "genre": "Animation", "rating": 5, "price": 1.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 20, "movie_id": 109, "title": "Interstellar", "genre": "Sci-Fi", "rating": 4, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    },
    {
        "customer_id": 2, "name": "Bob", "email": "bob@email.com", "city": "Los Angeles", "signup_date": "2023-02-20",
        "rentals": [
            {"rental_id": 4, "movie_id": 101, "title": "The Matrix", "genre": "Sci-Fi", "rating": 5, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 5, "movie_id": 110, "title": "The Room", "genre": "Drama", "rating": 2, "price": 1.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 21, "movie_id": 104, "title": "The Godfather", "genre": "Crime", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    },
    {
        "customer_id": 3, "name": "Charlie", "email": "charlie@email.com", "city": "Chicago", "signup_date": "2023-03-10",
        "rentals": [
            {"rental_id": 6, "movie_id": 104, "title": "The Godfather", "genre": "Crime", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 7, "movie_id": 105, "title": "Pulp Fiction", "genre": "Crime", "rating": 4, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 8, "movie_id": 106, "title": "The Dark Knight", "genre": "Action", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 22, "movie_id": 108, "title": "Parasite", "genre": "Thriller", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    },
    {
         "customer_id": 4, "name": "Diana", "email": "diana@email.com", "city": "Houston", "signup_date": "2023-04-05",
         "rentals": [
            {"rental_id": 9, "movie_id": 107, "title": "Spirited Away", "genre": "Animation", "rating": 5, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 10, "movie_id": 108, "title": "Parasite", "genre": "Thriller", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 23, "movie_id": 110, "title": "The Room", "genre": "Drama", "rating": 2, "price": 1.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
         ]
    },
    {
        "customer_id": 5, "name": "Eve", "email": "eve@email.com", "city": "Phoenix", "signup_date": "2023-05-12",
        "rentals": [
            {"rental_id": 11, "movie_id": 109, "title": "Interstellar", "genre": "Sci-Fi", "rating": 4, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 12, "movie_id": 110, "title": "The Room", "genre": "Drama", "rating": 2, "price": 1.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 13, "movie_id": 101, "title": "The Matrix", "genre": "Sci-Fi", "rating": 5, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 24, "movie_id": 102, "title": "Inception", "genre": "Sci-Fi", "rating": 4, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    },
    {
        "customer_id": 6, "name": "Frank", "email": "frank@email.com", "city": "New York", "signup_date": "2023-06-18",
        "rentals": [
            {"rental_id": 14, "movie_id": 102, "title": "Inception", "genre": "Sci-Fi", "rating": 4, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 15, "movie_id": 103, "title": "Toy Story", "genre": "Animation", "rating": 5, "price": 1.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    },
    {
        "customer_id": 7, "name": "Grace", "email": "grace@email.com", "city": "Los Angeles", "signup_date": "2023-07-22",
        "rentals": [
            {"rental_id": 16, "movie_id": 104, "title": "The Godfather", "genre": "Crime", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 17, "movie_id": 105, "title": "Pulp Fiction", "genre": "Crime", "rating": 4, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    },
    {
        "customer_id": 8, "name": "Heidi", "email": "heidi@email.com", "city": "Chicago", "signup_date": "2023-08-30",
        "rentals": [
            {"rental_id": 18, "movie_id": 106, "title": "The Dark Knight", "genre": "Action", "rating": 5, "price": 3.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 19, "movie_id": 107, "title": "Spirited Away", "genre": "Animation", "rating": 5, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"},
            {"rental_id": 25, "movie_id": 101, "title": "The Matrix", "genre": "Sci-Fi", "rating": 5, "price": 2.99, "rental_date": "2023-09-01", "return_date": "2023-09-05"}
        ]
    }
]

# Example to print one document
# print(json.dumps(customer_documents[0], indent=2))

In [9]:
from collections import defaultdict

# 1. Find all movies rented by a specific customer (e.g., Alice, customer_id = 1)
def get_movies_for_customer(customer_id):
    for doc in customer_documents:
        if doc['customer_id'] == customer_id:
            return [{"title": r['title'], "genre": r['genre'], "rental_date": r['rental_date']} for r in doc['rentals']]
    return []

print("1. Movies rented by Alice:")
print(get_movies_for_customer(1))


# 2. Find the top 5 most-rented movies
def get_top_5_movies():
    movie_counts = defaultdict(int)
    for doc in customer_documents:
        for rental in doc['rentals']:
            movie_counts[rental['title']] += 1
    
    # Sort by count descending, then take top 5
    sorted_movies = sorted(movie_counts.items(), key=lambda item: item[1], reverse=True)
    return sorted_movies[:5]

print("\n2. Top 5 Most Rented Movies:")
print(get_top_5_movies())


# 3. Compute total revenue per genre
def get_revenue_per_genre():
    genre_revenue = defaultdict(float)
    for doc in customer_documents:
        for rental in doc['rentals']:
            genre_revenue[rental['genre']] += rental['price']
            
    # Sort by revenue descending
    sorted_revenue = sorted(genre_revenue.items(), key=lambda item: item[1], reverse=True)
    # Format to 2 decimal places for readability
    return [(genre, round(rev, 2)) for genre, rev in sorted_revenue]

print("\n3. Total Revenue per Genre:")
print(get_revenue_per_genre())


# 4. Find customers who have never rented a movie rated below 3
def get_elite_customers():
    elite_customers = []
    for doc in customer_documents:
        # Check if ANY rental has a rating < 3. If so, flag them.
        has_bad_rental = any(rental['rating'] < 3 for rental in doc['rentals'])
        
        # If they haven't rented a bad movie, add them to our list
        if not has_bad_rental:
             elite_customers.append({"name": doc['name'], "email": doc['email']})
    return elite_customers

print("\n4. Customers who never rented a movie rated below 3:")
print(get_elite_customers())

1. Movies rented by Alice:
[{'title': 'The Matrix', 'genre': 'Sci-Fi', 'rental_date': '2023-09-01'}, {'title': 'Inception', 'genre': 'Sci-Fi', 'rental_date': '2023-09-01'}, {'title': 'Toy Story', 'genre': 'Animation', 'rental_date': '2023-09-01'}, {'title': 'Interstellar', 'genre': 'Sci-Fi', 'rental_date': '2023-09-01'}]

2. Top 5 Most Rented Movies:
[('The Matrix', 4), ('Inception', 3), ('The Room', 3), ('The Godfather', 3), ('Toy Story', 2)]

3. Total Revenue per Genre:
[('Sci-Fi', 31.91), ('Crime', 17.95), ('Animation', 9.96), ('Action', 7.98), ('Thriller', 7.98), ('Drama', 5.97)]

4. Customers who never rented a movie rated below 3:
[{'name': 'Alice', 'email': 'alice@email.com'}, {'name': 'Charlie', 'email': 'charlie@email.com'}, {'name': 'Frank', 'email': 'frank@email.com'}, {'name': 'Grace', 'email': 'grace@email.com'}, {'name': 'Heidi', 'email': 'heidi@email.com'}]


first and last query was easier, second and third was more difficult to write.
The Document Model shines in scenarios where data gets frequently read in its entirety and when writing the query should just be a matter of accessing nested records, whereas relational schemas can be trickier if you're not used to making queries out of linked data sets

In [10]:
from collections import Counter

# 1. Define the Nodes
nodes = {
    # Customers
    "c1": {"type": "customer", "name": "Alice"},
    "c2": {"type": "customer", "name": "Bob"},
    "c3": {"type": "customer", "name": "Charlie"},
    "c4": {"type": "customer", "name": "Diana"},
    
    # Movies
    "m101": {"type": "movie", "title": "The Matrix"},
    "m102": {"type": "movie", "title": "Inception"},
    "m103": {"type": "movie", "title": "Toy Story"},
    "m104": {"type": "movie", "title": "The Godfather"},
    "m105": {"type": "movie", "title": "Pulp Fiction"},
    "m110": {"type": "movie", "title": "The Room"},
    
    # Genres
    "g_scifi": {"type": "genre", "name": "Sci-Fi"},
    "g_anim": {"type": "genre", "name": "Animation"},
    "g_crime": {"type": "genre", "name": "Crime"},
    "g_drama": {"type": "genre", "name": "Drama"},
    
    # Cities
    "city_ny": {"type": "city", "name": "New York"},
    "city_la": {"type": "city", "name": "Los Angeles"}
}

# 2. Define the Edges (Relationships)
edges = [
    # Customer -> Movie (rented)
    ("c1", "rented", "m101"), ("c1", "rented", "m102"), ("c1", "rented", "m103"),
    ("c2", "rented", "m101"), ("c2", "rented", "m110"), ("c2", "rented", "m104"),
    ("c3", "rented", "m104"), ("c3", "rented", "m105"),
    ("c4", "rented", "m110"),
    
    # Movie -> Genre (belongs_to)
    ("m101", "belongs_to", "g_scifi"), ("m102", "belongs_to", "g_scifi"),
    ("m103", "belongs_to", "g_anim"),
    ("m104", "belongs_to", "g_crime"), ("m105", "belongs_to", "g_crime"),
    ("m110", "belongs_to", "g_drama"),
    
    # Customer -> City (lives_in)
    ("c1", "lives_in", "city_ny"),
    ("c2", "lives_in", "city_la")
]

in relational DB sql, to recommend something, we should write a lot of complex joins. But we can do this using graph model easily with connecting edges and nodes

In [ ]:
from datetime import datetime, timedelta

# Step 1: Create the DataFrame of 5,000 transactions
np.random.seed(42)
num_records = 5000

data = {
    'transaction_id': range(1, num_records + 1),
    'customer_id': np.random.randint(100, 999, num_records),
    'product_id': np.random.randint(1000, 5000, num_records),
    'amount': np.round(np.random.uniform(5.0, 500.0, num_records), 2),
    'payment_method': np.random.choice(['Credit Card', 'PayPal', 'Debit Card'], num_records),
    'timestamp': [datetime.now() - timedelta(days=np.random.randint(0, 365)) for _ in range(num_records)],
    'status': np.random.choice(['completed', 'refunded', 'pending'], num_records, p=[0.7, 0.1, 0.2])
}

df = pd.DataFrame(data)

# Step 2: Simulate OLTP Operations

# 1. Look up a single transaction (Point Query)
single_transaction = df[df['transaction_id'] == 1234]

# 2. Insert a new transaction (Append a row)
new_row = pd.DataFrame([{
    'transaction_id': 5001, 'customer_id': 202, 'product_id': 4040, 
    'amount': 45.99, 'payment_method': 'Credit Card', 
    'timestamp': datetime.now(), 'status': 'completed'
}])
df = pd.concat([df, new_row], ignore_index=True)

# 3. Update the status of a transaction
df.loc[df['transaction_id'] == 10, 'status'] = 'completed'

# 4. Consistency Check
# Making sure no data is corrupted (amounts must be positive, status must be valid)
valid_statuses = ['completed', 'refunded', 'pending']
is_consistent = (df['amount'] > 0).all() and df['status'].isin(valid_statuses).all()

In [12]:
# Step 3: Simulate OLAP Operations

# 1. Compute total revenue by payment method
# Grouping the whole table to see which payment method brings in the most money
revenue_by_method = df.groupby('payment_method')['amount'].sum()

# 2. Find the average transaction amount by month
# Converting timestamps to just the month and year, then averaging the amounts
df['month'] = df['timestamp'].dt.to_period('M')
avg_by_month = df.groupby('month')['amount'].mean()

# 3. Identify the top 10 customers by total spending
# Adding up everything each customer spent and grabbing the top 10
top_10_customers = df.groupby('customer_id')['amount'].sum().nlargest(10)

# 4. Compute the refund rate
# Counting how many transactions say 'refunded' and dividing by the total rows
refund_rate = (df['status'] == 'refunded').mean() * 100

in OLTP model, in specific time, we work on one user, update its data, delete or something. And for doing this efficiently we should have fast access to this users data, which is stored in row. that is why OLTP uses row major model. in the other hand, OLAP uses column-major format because we do not need one users data for analysis, instead, we need whole column which contains all the users data for aggregations. and this makes analysis faster and efficient

TASK 5

1.relational OLTP because each patient will have same fields in DB(name,sname,sickness, and soon). because we want to access and update one patient at a time, it is best to use OLTP

2.graph OLAP. For suggest and recommend, we need to connect multiple tables, infos and datas and graph is best for this purpose. OLAP will analyse data and make suggestions better

3.relational OLAP. because we should compute montly revenue and for doing this we need analyse processing.

4.graph OLAP

5.document OLTP